# Guardrail 11 — Cost

**Where it sits:** before any expensive operation (LLM call, embedding, retrieval over a large index) — and on the way out, for budget tracking.

**What it stops:**
  - token bombs (a 50,000-token query that costs $3 to answer)
  - budget overruns (one tenant spends the company's monthly LLM budget in an afternoon)
  - expensive-model overuse (calling GPT-4 for what GPT-4o-mini would answer)
  - embedding thrash (re-embedding the same query 1,000 times because no cache)
  - runaway loops (agent calls itself 200 times before noticing)

**Decision contract:** `{allow | rewrite | block, estimated_cost_usd, model_tier, reasons[]}`

**Self-contained:** inlines a toy pricing table and budget store. No imports from other folders.

## Step 1 — toy pricing table and budget store

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the same directory as this notebook (works in Jupyter)
_env_path = Path.cwd() / ".env"
if not _env_path.exists():
    _env_path = Path(__file__).parent / ".env" if "__file__" in globals() else _env_path
load_dotenv(_env_path, override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# LangChain primitives, all driven by .env
LLM_MODEL     = os.getenv("MINIMAX_MODEL", "MiniMax-M3")
LLM_BASE_URL  = os.getenv("MINIMAX_BASE_URL", "https://api.minimax.io/v1")
LLM_API_KEY   = os.getenv("MINIMAX_API_KEY", "")

llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=LLM_API_KEY or "sk-fake",   # placeholder if no key -- calls will fail loudly
    base_url=LLM_BASE_URL,
    temperature=0,
)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=LLM_API_KEY or "sk-fake",
    base_url=LLM_BASE_URL,
)

print(f"LLM configured:  model={LLM_MODEL}  base_url={LLM_BASE_URL}")
print(f"API key loaded:  {'yes ('+LLM_API_KEY[:8]+'...)' if LLM_API_KEY else 'NO -- calls will fail; toy fallbacks below are unaffected'}")

# A safe wrapper so toy guardrail tests below stay deterministic.
# If FAKE_LLM=1 (or no key), use the toy. Otherwise call the real one.
_USE_FAKE = os.getenv("FAKE_LLM", "1") == "1" or not LLM_API_KEY

def chat(prompt: str, system: str = None) -> str:
    """Invoke the LangChain ChatOpenAI. Returns .content."""
    if _USE_FAKE:
        raise RuntimeError("chat() called but FAKE_LLM=1 -- use the toy LLM in this notebook's tests")
    msgs = []
    if system:
        msgs.append(SystemMessage(content=system))
    msgs.append(HumanMessage(content=prompt))
    return llm.invoke(msgs).content


In [ ]:
import time

# Toy pricing — illustrative, not current
PRICING = {  # USD per 1M tokens
    "embedding-small":    {"in": 0.02,  "out": 0.00},
    "llm-cheap":          {"in": 0.15,  "out": 0.60},   # e.g. gpt-4o-mini
    "llm-mid":            {"in": 3.00,  "out": 12.00},  # e.g. gpt-4o
    "llm-big":            {"in": 15.00, "out": 60.00},  # e.g. gpt-4
}

class BudgetStore:
    """In-memory tenant spend tracker. Production = Redis with TTL."""
    def __init__(self):
        self.spent = {}  # tenant_id -> USD spent today

    def add(self, tenant_id: str, usd: float):
        self.spent[tenant_id] = self.spent.get(tenant_id, 0.0) + usd

    def remaining(self, tenant_id: str, cap_usd: float):
        return max(0.0, cap_usd - self.spent.get(tenant_id, 0.0))

budget = BudgetStore()
print("pricing loaded:", {k: f"${v['in']}/${v['out']} per 1M tok" for k,v in PRICING.items()})

## Step 2 — token estimation

In [ ]:
def estimate_tokens(text: str, chars_per_token: float = 4.0):
    """Cheap char-based estimate. Production: use the model's tokenizer."""
    return max(1, int(len(text) / chars_per_token))

def estimate_cost(model: str, in_tokens: int, out_tokens: int):
    p = PRICING[model]
    return (in_tokens / 1_000_000) * p["in"] + (out_tokens / 1_000_000) * p["out"]

## Step 3 — model tier routing

In [ ]:
def route_model(query: str,
                retrieved_chunks: list,
                difficulty_hint: str = None):
    """Pick the cheapest model that can plausibly answer.

    Heuristics (toy):
      - empty retrieval -> cheap (likely a refusal)
      - short query, simple keyword -> cheap
      - long query OR multi-hop -> big
      - difficulty_hint='hard' -> big regardless
    """
    if difficulty_hint == "hard":
        return "llm-big"
    if not retrieved_chunks:
        return "llm-cheap"   # nothing to ground on -> answer "I don't know" cheaply
    if len(query) < 80 and len(retrieved_chunks) <= 2:
        return "llm-cheap"
    if len(retrieved_chunks) >= 5 or len(query) > 400:
        return "llm-big"
    return "llm-mid"

## Step 4 — embedding cache

In [ ]:
class EmbeddingCache:
    """Tiny LRU-ish cache. Production = a vector DB or Redis."""
    def __init__(self, max_size=128):
        self.cache = {}
        self.max_size = max_size
        self.hits, self.misses = 0, 0

    def get_or_compute(self, text: str, embed_fn):
        key = text.strip().lower()
        if key in self.cache:
            self.hits += 1
            return self.cache[key], True
        if len(self.cache) >= self.max_size:
            self.cache.pop(next(iter(self.cache)))
        v = embed_fn(text)
        self.cache[key] = v
        self.misses += 1
        return v, False

    def stats(self):
        total = self.hits + self.misses
        return {"hits": self.hits, "misses": self.misses,
                "hit_rate": self.hits/total if total else 0}

embed_cache = EmbeddingCache()
def fake_embed(text):
    return [hash(w) % 97 / 97.0 for w in text.split()][:8] + [0.0]*8

# Demo: same query twice -> cache hit
for _ in range(2):
    embed_cache.get_or_compute("What is the capital of France?", fake_embed)
print("cache stats after 2 calls:", embed_cache.stats())

## Step 5 — the cost guardrail

In [ ]:
def cost_guard(query: str,
               identity: dict,
               retrieved_chunks: list,
               prompt_template: str,
               expected_out_tokens: int = 300,
               tenant_daily_cap_usd: float = 5.00,
               per_request_cap_usd: float = 0.50,
               max_in_tokens: int = 8000):
    """
    Returns the cheapest viable model tier that fits the budget, or
    blocks if the request itself or the tenant's day would be exceeded.
    """
    tenant = identity.get("tenant", "unknown")
    reasons = []

    # (a) assemble the full prompt and estimate in-tokens
    full_prompt = prompt_template.format(
        query=query,
        docs="\n".join(c.get("text","") for c in retrieved_chunks),
    )
    in_tokens = estimate_tokens(full_prompt)

    if in_tokens > max_in_tokens:
        return {"decision": "block",
                "estimated_cost_usd": None,
                "model_tier": None,
                "reasons": [f"in_tokens:{in_tokens}>{max_in_tokens}"]}

    # (b) pick model tier
    model = route_model(query, retrieved_chunks)

    # (c) estimate cost for THIS request
    cost = estimate_cost(model, in_tokens, expected_out_tokens)
    if cost > per_request_cap_usd:
        return {"decision": "block",
                "estimated_cost_usd": cost,
                "model_tier": model,
                "reasons": [f"per_request_cap:{cost:.4f}>{per_request_cap_usd}"]}

    # (d) check tenant's remaining daily budget
    remaining = budget.remaining(tenant, tenant_daily_cap_usd)
    if cost > remaining:
        return {"decision": "block",
                "estimated_cost_usd": cost,
                "model_tier": model,
                "reasons": [f"tenant_budget_exhausted:{remaining:.4f} left"]}

    # (e) reserve the spend (real systems: debit an atomic counter)
    budget.add(tenant, cost)
    reasons.append(f"model:{model}")
    reasons.append(f"est_cost_usd:{cost:.4f}")
    reasons.append(f"tenant_remaining_usd:{budget.remaining(tenant, tenant_daily_cap_usd):.4f}")

    return {"decision": "allow",
            "estimated_cost_usd": cost,
            "model_tier": model,
            "reasons": reasons}

## Step 6 — test cases

In [ ]:
PROMPT = "Answer the question.\nDOCS:\n{docs}\nQ: {query}\nA:"

identity = {"user_id": "u-alice", "tenant": "acme"}
tests = [
    ("short + 2 chunks",  "What is the capital of France?",
        [{"id":"d1","text":"The capital of France is Paris."},
         {"id":"d2","text":"France is in Europe."}]),

    ("long + 6 chunks",   "Compare the populations and capital status of France, Japan, and Brazil, citing each.",
        [{"id":f"d{i}","text":"text "*40} for i in range(6)]),

    ("empty retrieval",   "What is the meaning of life?", []),

    ("token bomb",        "x " * 20000,                  []),
]

for label, q, chunks in tests:
    r = cost_guard(q, identity, chunks, PROMPT, tenant_daily_cap_usd=5.0)
    print(f"\n=== {label} ===")
    print(f"  decision: {r['decision']}")
    print(f"  model   : {r['model_tier']}")
    print(f"  cost    : {r['estimated_cost_usd']}")
    print(f"  reasons : {r['reasons']}")

## Step 7 — tenant budget exhaustion

In [ ]:
# Spend acme's daily budget
budget.spent = {}
for i in range(20):
    cost_guard("What is the capital of France?", identity,
               [{"id":"d1","text":"The capital of France is Paris."}],
               PROMPT, tenant_daily_cap_usd=5.0)

print(f"acme has spent ${budget.spent.get('acme', 0):.4f} today")

# Next request -- should block
r = cost_guard("Any question", identity,
               [{"id":"d1","text":"x"}], PROMPT, tenant_daily_cap_usd=5.0)
print(f"\nnext request: decision={r['decision']}  reasons={r['reasons']}")

In [ ]:
### Real LangChain demo: cost guard as a Callback that tracks tokens

from langchain_community.callbacks import get_openai_callback

if not _USE_FAKE:
    with get_openai_callback() as cb:
        out = llm.invoke("What is the capital of France?").content
        print(f"response: {out!r}")
        print(f"tokens used: in={cb.prompt_tokens}  out={cb.completion_tokens}  total={cb.total_tokens}")
        print(f"estimated cost (USD): ${cb.total_cost:.6f}")
else:
    print("[FAKE_LLM=1 -- get_openai_callback skipped.]")


## Takeaways

- **Cost is a first-class guardrail, not an afterthought.** A single rogue query can spend more than a month's worth of legitimate traffic.
- **Three budgets, three checks.** Per-request cap (stops a single token bomb), per-tenant daily cap (stops a runaway script), per-month organization cap (stops an executive demo gone wrong).
- **Cheap-model-first is the default.** Routing 80% of traffic to `llm-cheap` and 20% to `llm-big` is the single largest cost win available. The classifier that decides which tier gets a request is itself a small ML problem.
- **Embedding caches pay for themselves in week one.** Anything with repeated queries (FAQ bots, dev environments) gets 30-70% hit rate within hours.
- **Token estimation is a floor, not a ceiling.** Char-based estimates are 80% accurate. Use the model's tokenizer when you can; reserve 20% headroom for variance.
- **Reserve the spend atomically.** A non-atomic check-then-spend has a TOCTOU race that ends with one tenant's budget subsidizing another's. Real systems use Redis INCRBY or a database row-level lock.

**Negative fixture checklist:** token bomb, per-request cap exceeded, tenant daily cap exhausted, expensive model for a trivial query, cache miss on a frequently-asked query. ✓